# 关于模型调用的invoke的使用
## 1.invoke的传参
### 1.1文本输入
举例:

In [ ]:
import os
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv
from langchain_core.messages import SystemMessage

#1.读取.env配置文件信息,相关的环境变量以.env文件中的优先
load_dotenv(verbose=True)
DEEPSEEK_API_KEY=os.getenv("DEEPSEEK_API_KEY")
DEEPSEEK_BASE_URL=os.getenv("DEEPSEEK_BASE_URL")
#2.模型初始化
model=init_chat_model(
    # model="deepseek-v4-flash",
    # model_provider="deepseek",
    model="deepseek:deepseek-v4-flash",
    api_key=DEEPSEEK_API_KEY,
    base_url=DEEPSEEK_BASE_URL
)
#3.模型调用
response=model.invoke("翻译如下的汉字：你好世界")
print(response)

### 1.2字典列表
举例1：

In [ ]:
messages=[
    {"role":"system","content":"你是一个专业的数学老师"},
    {"role":"user","content":"帮我解释一下什么事斐波那契数列"}
]
response=model.invoke(messages)
print(response)

举例2：涉及多轮对话

In [ ]:
messages=[
    {"role":"system","content":"你是一个专业的数学老师"},
    {"role":"user","content":"帮我解释一下什么事斐波那契数列"},
    {"role":"assistant","content":"3"},
    {"role":"user","content":"我刚才问了什么问题"}
]
response=model.invoke(messages)
print(response)

举例3：如果不传递历史，ai会失忆

In [ ]:
messages1 = [
{"role": "system", "content": "你是一个非常友好的AI助手"},
{"role": "user", "content": "你好，我叫小明"},
]
# 第一次对话
response1 = model.invoke(messages1)
# 打印响应
print(f"AI的回复1：{response1.content}")
messages2 = [
{"role": "user", "content": "我叫什么名字？"}
]
# 第二次对话
response2 = model.invoke(messages2)
# 打印响应
print(f"AI的回复2：{response2.content}")

作为对比，传递一下记忆

In [ ]:
conversation= [
{"role": "system", "content": "你是一个非常友好的AI助手"},
{"role": "user", "content": "你好，我叫小明"},
]
#第一次对话
response1=model.invoke(conversation)
print(f"ai的回复1:{response1.content}")
#添加记忆
conversation.append({"role":"assistant","content":response1.content})
conversation.append({"role":"user","content":"我叫什么名字"})
#第二次对话
response2=model.invoke(conversation)
print(f"ai的回复2:{response2.content}")

### 1.3消息对象列表

In [ ]:
from langchain_core.messages import SystemMessage,HumanMessage

messages=[
    SystemMessage(content="你是一个专业的数学老师"),
    HumanMessage(content="帮我解释一下斐波那契数列")
]
response=model.invoke(messages)
print(response)

举例2：传递记忆

In [ ]:
from langchain_core.messages import SystemMessage,HumanMessage,AIMessage

messages=[
    SystemMessage(content="你是一个专业的数学老师"),
    HumanMessage(content="帮我解释一下斐波那契数列")
]
#第一次对话
response1=model.invoke(messages)
print(f"ai第一次回复:{response1.content}")
#添加记忆
messages.append(AIMessage(content=response1.content))
messages.append(HumanMessage(content="我刚才问了什么问题"))
#第二次对话
response2=model.invoke(messages)
print(f"ai第二次回复:{response2.content}")

## 2.invoke的返回值
举例1:

In [ ]:
response=model.invoke([HumanMessage(content="2+3*2=?")])

In [ ]:
print(type(response))

In [ ]:
print(response)

美化输出

In [ ]:
from rich import print as rprint
rprint(response)

举例2

In [ ]:
response = model.invoke("用一句话解释什么是 AI")
# 1. 获取回复内容
print("AI 回复:", response.content)
# 2. 获取响应元数据
metadata = response.response_metadata
print(f"使用的模型: {metadata['model_name']}")
print(f"结束原因: {metadata['finish_reason']}")
print(f"模型提供商：{metadata['model_provider']}\n")
# 3. 获取 Token 使用情况
usage = metadata.get('token_usage', {})
print(f"输入 tokens: {usage.get('prompt_tokens')}")
print(f"输出 tokens: {usage.get('completion_tokens')}")
print(f"总计 tokens: {usage.get('total_tokens')}")
# 4. 获取消息 ID
print(f"消息 ID: {response.id}")